In [77]:
import warnings
warnings.filterwarnings("ignore")
import random
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import MinMaxScaler

In [78]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [79]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [80]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [81]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [82]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [83]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [84]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [85]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]

In [86]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [87]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [88]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [89]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 374)
y_train:  (139,)


(99, 376)

# Feature Selection: RENT

In [90]:
selected_features = ["shape_Sphericity",
                     "glrlm_HighGrayLevelRunEmphasis_PET_c04",
                     "shape_MajorAxisLength",
                     "LBP_102_PET"]

# Selecting features in the DataFrame
X_rent = X[selected_features]

In [91]:
# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Standardization

In [92]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = MinMaxScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [93]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [94]:
X_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.761164,16.969770,42.073251,0.000000
1,0.697049,15.598394,24.613845,0.000000
2,0.565792,17.334294,48.030294,0.000034
3,0.684364,14.009277,25.589900,0.000000
4,0.503142,21.202180,34.684750,0.000199
...,...,...,...,...
134,0.742102,16.110383,33.069705,0.000000
135,0.722918,21.249575,41.043692,0.000000
136,0.652963,14.873884,36.618802,0.000000
137,0.724255,21.648860,45.870392,0.000000


In [95]:
X_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.828413,0.487448,0.350904,0.000000
1,0.645006,0.388474,0.123069,0.000000
2,0.269537,0.513756,0.428640,0.066875
3,0.608721,0.273786,0.135806,0.000000
4,0.090323,0.792905,0.254489,0.386334
...,...,...,...,...
134,0.773882,0.425425,0.233413,0.000000
135,0.719008,0.796326,0.337469,0.000000
136,0.518897,0.336186,0.279727,0.000000
137,0.722831,0.825142,0.400455,0.000000


In [96]:
MAASTRO_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.668072,19.034156,50.002093,0.000026
1,0.669961,11.392444,41.753334,0.000167
2,0.624081,14.567421,44.375483,0.000057
3,0.577624,13.477331,46.115989,0.000000
4,0.630933,16.365554,54.394967,0.000000
...,...,...,...,...
94,0.671754,17.492295,34.218615,0.000000
95,0.632189,14.267900,51.046869,0.000000
96,0.645548,12.143752,50.417953,0.000000
97,0.727488,15.300229,44.901412,0.000000


In [97]:
MAASTRO_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.562115,0.636436,0.454371,0.050863
1,0.567520,0.084927,0.346729,0.324583
2,0.436277,0.314068,0.380947,0.110937
3,0.303383,0.235395,0.403660,0.000000
4,0.455879,0.443841,0.511695,0.000000
...,...,...,...,...
94,0.572648,0.525159,0.248406,0.000000
95,0.459472,0.292451,0.468005,0.000000
96,0.497683,0.139149,0.459798,0.000000
97,0.732079,0.366955,0.387810,0.000000


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [98]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 13:33:11,408] A new study created in memory with name: no-name-53d07cf2-a062-430e-a59f-b4d43c0bc3bd


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-15 13:33:11,573] A new study created in memory with name: no-name-28011d16-af58-4456-a8ef-f7dcf82e5b93


Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.676595744680851
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:33:11,569] Trial 0 finished with value: 0.6885812252956434 and parameters: {}. Best is trial 0 with value: 0.6885812252956434.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6885812252956434], datetime_start=datetime.datetime(2024, 4, 15, 13, 33, 11, 446106), datetime_complete=datetime.datetime(2024, 4, 15, 13, 33, 11, 569468), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6885812252956434


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2444511312501396
Fold 2 IBS: 0.19124878012978014
Fold 3 IBS: 0.2066529776721608
Fold 4 IBS: 0.19438765965098845
Fold 5 IBS: 0.19775948988265402
[I 2024-04-15 13:33:11,763] Trial 0 finished with value: 0.20690000771714462 and parameters: {}. Best is trial 0 with value: 0.20690000771714462.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.20690000771714462], datetime_start=datetime.datetime(2024, 4, 15, 13, 33, 11, 597857), datetime_complete=datetime.datetime(2024, 4, 15, 13, 33, 11, 763641), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.20690000771714462


In [99]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [100]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.689
train_ibs:  0.207


#### Test

In [101]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [102]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.532
IBS score: 0.29


In [103]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [104]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [105]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 13:33:11,886] A new study created in memory with name: no-name-771e5ef1-264f-4fff-8ce3-4335089a5a0a


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6075697211155379
Fold 2 C-index: 0.7073643410852714
Fold 3 C-index: 0.5446808510638298
Fold 4 C-index: 0.7243346007604563


[I 2024-04-15 13:33:11,991] A new study created in memory with name: no-name-659aa205-7bd7-49b2-84bb-3323b19007af


Fold 5 C-index: 0.6545064377682404
[I 2024-04-15 13:33:11,988] Trial 0 finished with value: 0.6476911903586672 and parameters: {}. Best is trial 0 with value: 0.6476911903586672.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6476911903586672], datetime_start=datetime.datetime(2024, 4, 15, 13, 33, 11, 916034), datetime_complete=datetime.datetime(2024, 4, 15, 13, 33, 11, 988073), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6476911903586672


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709895823235
Fold 2 IBS: 0.23203988238189202
Fold 3 IBS: 0.22898186794169834
Fold 4 IBS: 0.24197476877861912
Fold 5 IBS: 0.2293955911760588
[I 2024-04-15 13:33:12,122] Trial 0 finished with value: 0.23592784184730015 and parameters: {}. Best is trial 0 with value: 0.23592784184730015.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784184730015], datetime_start=datetime.datetime(2024, 4, 15, 13, 33, 12, 25431), datetime_complete=datetime.datetime(2024, 4, 15, 13, 33, 12, 122303), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784184730015


In [106]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [107]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.648
train_ibs:  0.236


#### Test

In [108]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [109]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [110]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [111]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 13:33:12,254] A new study created in memory with name: no-name-e6cba13a-2bdf-48df-a818-4c2ad3f45165


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872


[I 2024-04-15 13:33:12,480] A new study created in memory with name: no-name-b5702b19-36ab-4c81-bd31-f98138e9d60d


Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:33:12,474] Trial 0 finished with value: 0.6910259146234387 and parameters: {}. Best is trial 0 with value: 0.6910259146234387.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6910259146234387], datetime_start=datetime.datetime(2024, 4, 15, 13, 33, 12, 284693), datetime_complete=datetime.datetime(2024, 4, 15, 13, 33, 12, 474642), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6910259146234387


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24327474589214967
Fold 2 IBS: 0.19124193562772077
Fold 3 IBS: 0.20659154302819144
Fold 4 IBS: 0.19431608966437242
Fold 5 IBS: 0.1985649880350335
[I 2024-04-15 13:33:12,721] Trial 0 finished with value: 0.20679786044949355 and parameters: {}. Best is trial 0 with value: 0.20679786044949355.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.20679786044949355], datetime_start=datetime.datetime(2024, 4, 15, 13, 33, 12, 515995), datetime_complete=datetime.datetime(2024, 4, 15, 13, 33, 12, 721195), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.20679786044949355


In [112]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [113]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.691
train_ibs:  0.207


#### Test 

In [114]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [115]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.288


In [116]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [117]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 13:33:12,891] A new study created in memory with name: no-name-01fc6c22-aef1-431c-a898-837d737eea3a


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:33:13,113] Trial 0 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6910259146234387.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:33:13,392] Trial 1 finished with value: 0.6925763022203378 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.6925763022203378.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:33:13,529] Trial 2 finished with value: 0.6634233210699454 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:33:17,285] Trial 24 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.3856495824719821}. Best is trial 1 with value: 0.6925763022203378.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:33:17,438] Trial 25 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.8000279351376395}. Best is trial 1 with value: 0.6925763022203378.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:33:17,585] Trial 26 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.43696099753129897

Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:33:20,533] Trial 48 finished with value: 0.6687465149862952 and parameters: {'l1_ratio': 0.2629781061488513}. Best is trial 1 with value: 0.6925763022203378.
Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.7224334600760456
Fold 5 C-index: 0.6738197424892703
[I 2024-04-15 13:33:20,601] Trial 49 finished with value: 0.6607686193974597 and parameters: {'l1_ratio': 0.005040986123852953}. Best is trial 1 with value: 0.6925763022203378.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:33:20,791] Trial 50 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.6064718647459368}. Best is trial 1 with value: 0.6925763022203378.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.24346704604704877
Fold 2 IBS: 0.19111872569472418
Fold 3 IBS: 0.20600858708655184
Fold 4 IBS: 0.1941307581219963
Fold 5 IBS: 0.19818777104825638
[I 2024-04-15 13:33:28,519] Trial 0 finished with value: 0.2065825775997155 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.2065825775997155.
Fold 1 IBS: 0.2436561534893646
Fold 2 IBS: 0.22555923310558854
Fold 3 IBS: 0.20542452204756903
Fold 4 IBS: 0.19394964500544354
Fold 5 IBS: 0.1979325614057397
[I 2024-04-15 13:33:28,660] Trial 1 finished with value: 0.2133044230107411 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.2065825775997155.
Fold 1 IBS: 0.24369121579073802
Fold 2 IBS: 0.22695059901402637
Fold 3 IBS: 0.22875723206890197
Fold 4 IBS: 0.23582102158769938
Fold 5 IBS: 0.19789779493852902
[I 2024-04-15 13:33:28,766] Trial 2 finished with value: 0.22662357267997896 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.2065825775997155

Fold 2 IBS: 0.22675322787093563
Fold 3 IBS: 0.2287523143790381
Fold 4 IBS: 0.23559026194922902
Fold 5 IBS: 0.1979102750890993
[I 2024-04-15 13:33:32,118] Trial 25 finished with value: 0.22654479628683327 and parameters: {'l1_ratio': 0.2354015085405634}. Best is trial 15 with value: 0.206410265144237.
Fold 1 IBS: 0.2437260804372543
Fold 2 IBS: 0.22802890772770576
Fold 3 IBS: 0.2287898443018471
Fold 4 IBS: 0.23709312224879853
Fold 5 IBS: 0.22598028522086833
[I 2024-04-15 13:33:32,350] Trial 26 finished with value: 0.2327236479872948 and parameters: {'l1_ratio': 0.1802457489939742}. Best is trial 15 with value: 0.206410265144237.
Fold 1 IBS: 0.24355884024356472
Fold 2 IBS: 0.19107607756683415
Fold 3 IBS: 0.20580695943058178
Fold 4 IBS: 0.1940669657408476
Fold 5 IBS: 0.19811918663666345
[I 2024-04-15 13:33:32,526] Trial 27 finished with value: 0.20652560592369834 and parameters: {'l1_ratio': 0.58416616744615}. Best is trial 15 with value: 0.206410265144237.
Fold 1 IBS: 0.24369023977189563


Fold 3 IBS: 0.2055675332348129
Fold 4 IBS: 0.19396255820054176
Fold 5 IBS: 0.1979615210358094
[I 2024-04-15 13:33:36,413] Trial 50 finished with value: 0.20642549568411833 and parameters: {'l1_ratio': 0.3347781725956745}. Best is trial 38 with value: 0.20640346841169058.
Fold 1 IBS: 0.2436267647873312
Fold 2 IBS: 0.1909994247303081
Fold 3 IBS: 0.2055545740659544
Fold 4 IBS: 0.19395772544342527
Fold 5 IBS: 0.19796190795053162
[I 2024-04-15 13:33:36,593] Trial 51 finished with value: 0.20642007939551013 and parameters: {'l1_ratio': 0.33194260491009325}. Best is trial 38 with value: 0.20640346841169058.
Fold 1 IBS: 0.24371349079799468
Fold 2 IBS: 0.2262549590373793
Fold 3 IBS: 0.2287414448160795
Fold 4 IBS: 0.19392825456918603
Fold 5 IBS: 0.197921385531121
[I 2024-04-15 13:33:36,758] Trial 52 finished with value: 0.2181119069503521 and parameters: {'l1_ratio': 0.25669632246926505}. Best is trial 38 with value: 0.20640346841169058.
Fold 1 IBS: 0.2436833011970229
Fold 2 IBS: 0.2274679232805

Fold 1 IBS: 0.24366472796576338
Fold 2 IBS: 0.1910152609366343
Fold 3 IBS: 0.20553283565862954
Fold 4 IBS: 0.19394840577468178
Fold 5 IBS: 0.1979597961934454
[I 2024-04-15 13:33:40,285] Trial 75 finished with value: 0.2064242053058309 and parameters: {'l1_ratio': 0.34830106369663216}. Best is trial 38 with value: 0.20640346841169058.
Fold 1 IBS: 0.24364576255356746
Fold 2 IBS: 0.19099718469457486
Fold 3 IBS: 0.20554697858455104
Fold 4 IBS: 0.19395604846742018
Fold 5 IBS: 0.19794632560411193
[I 2024-04-15 13:33:40,461] Trial 76 finished with value: 0.2064184599808451 and parameters: {'l1_ratio': 0.3098663919454113}. Best is trial 38 with value: 0.20640346841169058.
Fold 1 IBS: 0.24369113096869818
Fold 2 IBS: 0.22695120380877024
Fold 3 IBS: 0.228757247658077
Fold 4 IBS: 0.2358217297094029
Fold 5 IBS: 0.19789780014323363
[I 2024-04-15 13:33:40,569] Trial 77 finished with value: 0.2266238224576364 and parameters: {'l1_ratio': 0.22690277246993334}. Best is trial 38 with value: 0.20640346841

In [118]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [119]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.693
train_ibs:  0.206


#### Test

In [120]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [121]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.28621072101688444)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.29543822482619353)

test_ibs:  0.288


In [122]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [123]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 13:33:45,426] A new study created in memory with name: no-name-d11239e4-f9dd-492d-b0f6-fbffe68afd73


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6454183266932271
Fold 2 C-index: 0.7945736434108527
Fold 3 C-index: 0.6723404255319149
Fold 4 C-index: 0.7281368821292775
Fold 5 C-index: 0.648068669527897
[I 2024-04-15 13:33:47,569] Trial 0 finished with value: 0.6977075894586338 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6977075894586338.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7906976744186046
Fold 3 C-index: 0.7276595744680852
Fold 4 C-index: 0.7338403041825095
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:33:48,976] Trial 1 finished with value: 0.7128781680014569 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 4 C-index: 0.779467680608365
Fold 5 C-index: 0.7060085836909872
[I 2024-04-15 13:34:01,112] Trial 16 finished with value: 0.7424649927687818 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 7, 'min_samples_leaf': 16, 'max_depth': 7, 'n_estimators': 7, 'oob_score': True, 'max_samples': 0.9632036286697128, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19855793167729388, 'warm_start': True}. Best is trial 16 with value: 0.7424649927687818.
Fold 1 C-index: 0.6812749003984063
Fold 2 C-index: 0.7829457364341085
Fold 3 C-index: 0.7574468085106383
Fold 4 C-index: 0.7110266159695817
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 13:34:01,169] Trial 17 finished with value: 0.7187276534642637 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 8, 'min_samples_leaf': 16, 'max_depth': 7, 'n_estimators': 4, 'oob_score': True, 'max_samples': 0.9920927723346319, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.3637168382811743, 'warm_start': True}. Best is trial 1

Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.8178294573643411
Fold 3 C-index: 0.8425531914893617
Fold 4 C-index: 0.8136882129277566
Fold 5 C-index: 0.7639484978540773
[I 2024-04-15 13:34:07,286] Trial 31 finished with value: 0.7735002862697369 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 7, 'max_depth': 20, 'n_estimators': 379, 'oob_score': True, 'max_samples': 0.6547593902381754, 'max_features': None, 'min_weight_fraction_leaf': 0.01653304213676029, 'warm_start': True}. Best is trial 29 with value: 0.7751733627229969.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.8178294573643411
Fold 3 C-index: 0.8425531914893617
Fold 4 C-index: 0.8174904942965779
Fold 5 C-index: 0.7639484978540773
[I 2024-04-15 13:34:08,114] Trial 32 finished with value: 0.7726671170454932 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 7, 'max_depth': 20, 'n_estimators': 390, 'oob_score': True, 'max_samples': 0.7011070731694715,

...

Fold 1 IBS: 0.22612682993385486
Fold 2 IBS: 0.17223298934748948
Fold 3 IBS: 0.20782358499736456
Fold 4 IBS: 0.20181112716562463
Fold 5 IBS: 0.21350833459590543
[I 2024-04-15 13:36:07,643] Trial 32 finished with value: 0.2043005732080478 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 11, 'min_samples_leaf': 13, 'max_depth': 8, 'n_estimators': 316, 'oob_score': False, 'max_samples': 0.8753375341925101, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.2230974552568295}. Best is trial 1 with value: 0.20384699429436565.
Fold 1 IBS: 0.22177687649658975
Fold 2 IBS: 0.17630377861515312
Fold 3 IBS: 0.2122309287307206
Fold 4 IBS: 0.2017574245411477
Fold 5 IBS: 0.21470545803951144
[I 2024-04-15 13:36:09,873] Trial 33 finished with value: 0.20535489328462447 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 307, 'oob_score': False, 'max_samples': 0.8780403780133842, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.23790435513187447
Fold 2 IBS: 0.16917310562036944
Fold 3 IBS: 0.2032724536335342
Fold 4 IBS: 0.20275263571784216
Fold 5 IBS: 0.21198800508925086
[I 2024-04-15 13:36:28,186] Trial 48 finished with value: 0.20501811103857426 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 14, 'min_samples_leaf': 13, 'max_depth': 10, 'n_estimators': 369, 'oob_score': False, 'max_samples': 0.862646612475437, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.18882619674434536}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.21643739431200149
Fold 2 IBS: 0.18002210069568697
Fold 3 IBS: 0.20963559196115233
Fold 4 IBS: 0.20201711337877098
Fold 5 IBS: 0.21711926927169345
[I 2024-04-15 13:36:28,734] Trial 49 finished with value: 0.20504629392386103 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 12, 'min_samples_leaf': 19, 'max_depth': 7, 'n_estimators': 68, 'oob_score': False, 'max_samples': 0.712240935894755, 'max_features': 'sqrt', 'min_weight_fraction

Fold 5 IBS: 0.21281753768850425
[I 2024-04-15 13:36:41,577] Trial 63 finished with value: 0.20478734721351177 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 16, 'min_samples_leaf': 12, 'max_depth': 6, 'n_estimators': 145, 'oob_score': False, 'max_samples': 0.5771231216606082, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.05763095148942208}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.21904071817956408
Fold 2 IBS: 0.17322770907939164
Fold 3 IBS: 0.21071337338690455
Fold 4 IBS: 0.20198953222034577
Fold 5 IBS: 0.21531568867352077
[I 2024-04-15 13:36:42,040] Trial 64 finished with value: 0.20405740430794536 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 2, 'n_estimators': 74, 'oob_score': False, 'max_samples': 0.6331344922813995, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1612709153840626}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.2211019166635484
Fold 2 IBS: 0.175

Fold 5 IBS: 0.21518309439089522
[I 2024-04-15 13:36:50,592] Trial 79 finished with value: 0.2023912644271136 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 6, 'n_estimators': 25, 'oob_score': False, 'max_samples': 0.7940170614934956, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19696931079449848}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.2449767955817733
Fold 2 IBS: 0.172095460075217
Fold 3 IBS: 0.21463433607213933
Fold 4 IBS: 0.21510140082957746
Fold 5 IBS: 0.21527266884222518
[I 2024-04-15 13:36:50,846] Trial 80 finished with value: 0.21241613228018644 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 10, 'max_depth': 6, 'n_estimators': 28, 'oob_score': False, 'max_samples': 0.8170871700419534, 'max_features': None, 'min_weight_fraction_leaf': 0.21125680575069572}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.22348296120259534
Fold 2 IBS: 0.16980207

Fold 2 IBS: 0.1697323509215535
Fold 3 IBS: 0.20905255797983618
Fold 4 IBS: 0.2258672184082545
Fold 5 IBS: 0.2152407835992728
[I 2024-04-15 13:36:56,997] Trial 95 finished with value: 0.21784360855002127 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 11, 'min_samples_leaf': 1, 'max_depth': 12, 'n_estimators': 25, 'oob_score': False, 'max_samples': 0.8547756485942715, 'max_features': None, 'min_weight_fraction_leaf': 0.17594983744388534}. Best is trial 86 with value: 0.20204409612264693.
Fold 1 IBS: 0.22065985212578162
Fold 2 IBS: 0.1724043068676393
Fold 3 IBS: 0.212876495767763
Fold 4 IBS: 0.20196603399913246
Fold 5 IBS: 0.21590968295775706
[I 2024-04-15 13:36:57,348] Trial 96 finished with value: 0.2047632743436147 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 7, 'min_samples_leaf': 15, 'max_depth': 13, 'n_estimators': 61, 'oob_score': False, 'max_samples': 0.7614906038154946, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.140922870660813}. Best is tria

In [124]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [125]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.83
train_ibs:  0.202


#### Test

In [126]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [127]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=18, max_leaf_nodes=11,
                     max_samples=0.9059465751594485, min_samples_leaf=1,
                     min_weight_fraction_leaf=0.029068528038335394,
                     n_estimators=318, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.544


RandomSurvivalForest(max_depth=13, max_features='log2', max_leaf_nodes=11,
                     max_samples=0.7105780352803006, min_samples_leaf=11,
                     min_samples_split=13,
                     min_weight_fraction_leaf=0.2021626029496649,
                     n_estimators=31, random_state=123)

test_ibs:  0.247


In [128]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [129]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 13:36:59,432] A new study created in memory with name: no-name-88b4cc1f-7ed9-4999-853d-2cf37282b8a3


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.7829457364341085
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.7490494296577946
Fold 5 C-index: 0.6909871244635193
[I 2024-04-15 13:36:59,931] Trial 0 finished with value: 0.7329375617815092 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7329375617815092.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:37:01,115] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:37:11,732] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 20, 'n_estimators': 246, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.12362405012814498, 'min_weight_fraction_leaf': 0.18949444225079395}. Best is trial 8 with value: 0.7362139149697521.
Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.7906976744186046
Fold 3 C-index: 0.7872340425531915
Fold 4 C-index: 0.7604562737642585
Fold 5 C-index: 0.6909871244635193
[I 2024-04-15 13:37:12,241] Trial 17 finished with value: 0.7333650628805521 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 9, 'n_estimators': 362, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8640754037508087, 'min_weight_fraction_leaf': 0.08300984976320

Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.7945736434108527
Fold 3 C-index: 0.7914893617021277
Fold 4 C-index: 0.7566539923954373
Fold 5 C-index: 0.6866952789699571
[I 2024-04-15 13:37:21,502] Trial 31 finished with value: 0.7301852441402965 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 9, 'min_samples_leaf': 10, 'max_depth': 11, 'n_estimators': 288, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8780867392507613, 'min_weight_fraction_leaf': 0.08053342835298444}. Best is trial 8 with value: 0.7362139149697521.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7868217054263565
Fold 3 C-index: 0.7829787234042553
Fold 4 C-index: 0.7490494296577946
Fold 5 C-index: 0.6909871244635193
[I 2024-04-15 13:37:21,929] Trial 32 finished with value: 0.7270669981840107 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.2369623605519169
Fold 2 IBS: 0.2015664827514855
Fold 3 IBS: 0.2044196732107329
Fold 4 IBS: 0.2152719150958778
Fold 5 IBS: 0.2123358137500061
[I 2024-04-15 13:38:56,389] Trial 30 finished with value: 0.21411124907200385 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.20923338228172916.
Fold 1 IBS: 0.23451695205118067
Fold 2 IBS: 0.19751010562426458
Fold 3 IBS: 0.2027198084019687
Fold 4 IBS: 0.21411412044401185
Fold 5 IBS: 0.2111761169753505
[I 2024-04-15 13:38:59,313] Trial 31 finished with value: 0.21200742069935527 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 409, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.923

Fold 1 IBS: 0.23247724033039832
Fold 2 IBS: 0.18748554961121114
Fold 3 IBS: 0.2090170455982855
Fold 4 IBS: 0.207742375469341
Fold 5 IBS: 0.20889347688505747
[I 2024-04-15 13:39:45,175] Trial 45 finished with value: 0.2091231375788587 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 14, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 144, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8505483502716148, 'min_weight_fraction_leaf': 0.09794633817349793}. Best is trial 45 with value: 0.2091231375788587.
Fold 1 IBS: 0.2314286160573986
Fold 2 IBS: 0.1860889745854672
Fold 3 IBS: 0.20638267011548084
Fold 4 IBS: 0.20884587453459477
Fold 5 IBS: 0.20805701088484838
[I 2024-04-15 13:39:46,285] Trial 46 finished with value: 0.20816062923555795 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 16, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 138, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.77092

Fold 2 IBS: 0.21593994734134395
Fold 3 IBS: 0.22177987321764975
Fold 4 IBS: 0.22790863488656224
Fold 5 IBS: 0.2205529164521157
[I 2024-04-15 13:40:07,918] Trial 60 finished with value: 0.22476557086692814 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 16, 'min_samples_leaf': 3, 'max_depth': 20, 'n_estimators': 74, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.6041879821929975, 'min_weight_fraction_leaf': 0.1056236625824671}. Best is trial 46 with value: 0.20816062923555795.
Fold 1 IBS: 0.23781793031064252
Fold 2 IBS: 0.1862495208476139
Fold 3 IBS: 0.20248456745784213
Fold 4 IBS: 0.21234711020159586
Fold 5 IBS: 0.2076972639565933
[I 2024-04-15 13:40:09,451] Trial 61 finished with value: 0.20931927855485752 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 2, 'max_depth': 19, 'n_estimators': 197, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7101126073432302, 'min_weight_fraction_

Fold 1 IBS: 0.23693321208405524
Fold 2 IBS: 0.19000973925047765
Fold 3 IBS: 0.20243881773088923
Fold 4 IBS: 0.2110667431713562
Fold 5 IBS: 0.20765848231427028
[I 2024-04-15 13:40:21,903] Trial 75 finished with value: 0.20962139891020973 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 18, 'n_estimators': 104, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.49599352625156035, 'min_weight_fraction_leaf': 0.03138093454428739}. Best is trial 71 with value: 0.20716724376288945.
Fold 1 IBS: 0.2375162774090104
Fold 2 IBS: 0.21696007860868566
Fold 3 IBS: 0.2199964189465905
Fold 4 IBS: 0.2298197447784768
Fold 5 IBS: 0.2208032877666601
[I 2024-04-15 13:40:22,612] Trial 76 finished with value: 0.2250191615018847 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 14, 'min_samples_leaf': 1, 'max_depth': 17, 'n_estimators': 81, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.6498539

Fold 1 IBS: 0.23900229487413618
Fold 2 IBS: 0.1829989350900894
Fold 3 IBS: 0.20504008460974205
Fold 4 IBS: 0.21135329415618298
Fold 5 IBS: 0.20658610615550185
[I 2024-04-15 13:40:41,642] Trial 90 finished with value: 0.20899614297713048 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 1, 'max_depth': 16, 'n_estimators': 95, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7812194815989179, 'min_weight_fraction_leaf': 0.052583924158953604}. Best is trial 71 with value: 0.20716724376288945.
Fold 1 IBS: 0.23336269917153504
Fold 2 IBS: 0.18793910685243745
Fold 3 IBS: 0.20227191981491185
Fold 4 IBS: 0.2126322523584538
Fold 5 IBS: 0.20891055719221097
[I 2024-04-15 13:40:43,329] Trial 91 finished with value: 0.2090233070779098 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 170, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7

In [130]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [131]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.764
train_ibs:  0.207


#### Test

In [132]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [133]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=6, max_features=None, max_leaf_nodes=13,
                   max_samples=0.9529576562335005, min_samples_split=15,
                   min_weight_fraction_leaf=0.05156899271012762,
                   n_estimators=376, random_state=123, warm_start=True)

C-index score: 0.543


ExtraSurvivalTrees(max_depth=19, max_features=None, max_leaf_nodes=17,
                   max_samples=0.6638490929451707, min_samples_leaf=2,
                   min_samples_split=19,
                   min_weight_fraction_leaf=0.052914450220883674,
                   n_estimators=60, oob_score=True, random_state=123)

IBS: 0.256


In [134]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [135]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 13:40:54,327] A new study created in memory with name: no-name-7cbfa91a-27d6-4980-8745-28ba33eaea14


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:41:04,641] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:41:10,108] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:44:21,526] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7079199845805899.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:44:43,521] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:49:37,985] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.7079199845805899.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 13:49:48,237] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-15 15:36:55,671] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.918589848048704, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.1366452847323028, 'n_estimators': 388, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.18967222528443176, 'max_features': 'auto', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 18}. Best is trial 53 with value: 0.2348064399929884.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 15:38:28,058] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7513175598857118, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.357604

Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 15:51:01,372] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8952643616873973, 'learning_rate': 0.05359198006915804, 'dropout_rate': 0.6509686174553241, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.713342732410399, 'min_weight_fraction_leaf': 0.037327349410482574, 'max_features': 'auto', 'min_impurity_decrease': 2.2946753237767036e-06, 'validation_fraction': 0.8670144195682054, 'min_samples_split': 9, 'max_leaf_nodes': 20, 'min_samples_leaf': 14, 'max_depth': 18}. Best is trial 64 with value: 0.23475649727610257.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 15:51:40,487] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9559398584951578, 'learning_rate': 0.001312025546225645, 'dropout_rate': 0.8046

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-15 16:02:53,900] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9040601608260395, 'learning_rate': 0.008028762030775153, 'dropout_rate': 0.1661055390191164, 'n_estimators': 412, 'criterion': 'squared_error', 'ccp_alpha': 0.5920167940309409, 'min_weight_fraction_leaf': 0.20760648939509768, 'max_features': 1, 'min_impurity_decrease': 1.2858411523836384e-06, 'validation_fraction': 0.7519570151291436, 'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 64 with value: 0.23475649727610257.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 16:03:31,657] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9412481314247185, 'learning_rate': 0.003888190949209832, 'dropout_rate': 0.625165508

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 16:15:17,066] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7762725560789899, 'learning_rate': 0.0030671608519508686, 'dropout_rate': 0.2500967923284591, 'n_estimators': 479, 'criterion': 'squared_error', 'ccp_alpha': 1.2742275973203492, 'min_weight_fraction_leaf': 0.27772784044341225, 'max_features': 'auto', 'min_impurity_decrease': 0.0011701047770450368, 'validation_fraction': 0.9552938036430088, 'min_samples_split': 13, 'max_leaf_nodes': 19, 'min_samples_leaf': 17, 'max_depth': 2}. Best is trial 85 with value: 0.23301282639832266.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 16:17:01,351] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9240076064991638, 'learning_rate': 0.036372907201929795, 'dropout_rate': 0.166

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.22939559304809248
[I 2024-04-15 16:32:48,689] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9084360842440081, 'learning_rate': 0.06808689488183181, 'dropout_rate': 0.25443633448028735, 'n_estimators': 437, 'criterion': 'squared_error', 'ccp_alpha': 0.5386043790221791, 'min_weight_fraction_leaf': 0.03403879613376609, 'max_features': 'auto', 'min_impurity_decrease': 8.69596280226011e-07, 'validation_fraction': 0.9221483446288596, 'min_samples_split': 20, 'max_leaf_nodes': 12, 'min_samples_leaf': 16, 'max_depth': 2}. Best is trial 85 with value: 0.23301282639832266.


* Best trial for IBS: 
 FrozenTrial(number=85, state=TrialState.COMPLETE, values=[0.23301282639832266], datetime_start=datetime.datetime(2024, 4, 15, 16, 9, 40, 66714), datetime_complete=datetime.datetime(2024, 4, 15, 16, 11, 14, 702735), params={'subsample': 0.9481727373988897, 'learning_rate': 0.020229752590967

In [136]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [137]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.713
train_ibs:  0.233


#### Test

In [138]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [139]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.022737959076050005,
                                 dropout_rate=0.8285090704895912,
                                 learning_rate=0.0513322635401625, max_depth=2,
                                 max_features='sqrt', max_leaf_nodes=10,
                                 min_impurity_decrease=0.002070496973827786,
                                 min_samples_leaf=15, min_samples_split=19,
                                 min_weight_fraction_leaf=0.3101145382804477,
                                 n_estimators=485, random_state=123,
                                 subsample=0.45214538811231725,
                                 validation_fraction=0.9071319756271838)

C-index score: 0.532


GradientBoostingSurvivalAnalysis(ccp_alpha=0.01138981446692314,
                                 criterion='squared_error',
                                 dropout_rate=0.2479619862207481,
                                 learning_rate=0.02022975259096714, max_depth=8,
                                 max_features='auto', max_leaf_nodes=20,
                                 min_impurity_decrease=9.61920586779085e-07,
                                 min_samples_leaf=18, min_samples_split=14,
                                 min_weight_fraction_leaf=0.026767353450782638,
                                 n_estimators=438, random_state=123,
                                 subsample=0.9481727373988897,
                                 validation_fraction=0.973754058089352)

IBS: 0.229


In [140]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [141]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 16:33:17,622] A new study created in memory with name: no-name-fafcec89-1782-4783-a48b-4fe0e440c784


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.751937984496124
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6008583690987125
[I 2024-04-15 16:33:19,237] Trial 0 finished with value: 0.6238037606018434 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6238037606018434.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.751937984496124
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6008583690987125
[I 2024-04-15 16:33:33,043] Trial 1 finished with value: 0.6238037606018434 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6238037606018434.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7441860465116279
Fold 3 C-index: 0.6085106382978723
Fold 4 

Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.7063829787234043
Fold 4 C-index: 0.6958174904942965
Fold 5 C-index: 0.6394849785407726
[I 2024-04-15 16:35:22,767] Trial 19 finished with value: 0.6679528899161283 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9820749239416067, 'n_estimators': 431, 'learning_rate': 0.08300323114610605}. Best is trial 14 with value: 0.6686270138087715.
Fold 1 C-index: 0.5537848605577689
Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.6723404255319149
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.630901287553648
[I 2024-04-15 16:35:31,962] Trial 20 finished with value: 0.6540975657381868 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.7933084651006226, 'n_estimators': 438, 'learning_rate': 0.0796244585080611}. Best is trial 14 with value: 0.6686270138087715.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7441860465116279
Fold 3 C-index: 0.7063829787234043
Fol

Fold 1 C-index: 0.549800796812749
Fold 2 C-index: 0.751937984496124
Fold 3 C-index: 0.7021276595744681
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6351931330472103
[I 2024-04-15 16:37:26,661] Trial 38 finished with value: 0.6616522189686197 and parameters: {'subsample': 0.23557094034920928, 'dropout_rate': 0.777571897600354, 'n_estimators': 393, 'learning_rate': 0.07844703170656485}. Best is trial 22 with value: 0.6706655505801468.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.751937984496124
Fold 3 C-index: 0.6468085106382979
Fold 4 C-index: 0.6577946768060836
Fold 5 C-index: 0.6180257510729614
[I 2024-04-15 16:37:36,238] Trial 39 finished with value: 0.6488576077102631 and parameters: {'subsample': 0.31440461040323664, 'dropout_rate': 0.6881141432789704, 'n_estimators': 459, 'learning_rate': 0.01033884503473171}. Best is trial 22 with value: 0.6706655505801468.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.6212765957446809
Fold

...

Fold 2 IBS: 0.16248142823028444
Fold 3 IBS: 0.23157700988647345
Fold 4 IBS: 0.284982956349929
Fold 5 IBS: 0.29378159734341147
[I 2024-04-15 16:43:28,725] Trial 19 finished with value: 0.267707812129161 and parameters: {'subsample': 0.36536815139162093, 'dropout_rate': 0.9906028341543912, 'n_estimators': 341, 'learning_rate': 0.09290979487234556}. Best is trial 15 with value: 0.21526209245288125.
Fold 1 IBS: 0.24752088807102174
Fold 2 IBS: 0.2068435694297299
Fold 3 IBS: 0.21900644273712963
Fold 4 IBS: 0.22620957065689154
Fold 5 IBS: 0.21957258967822554
[I 2024-04-15 16:43:29,416] Trial 20 finished with value: 0.22383061211459965 and parameters: {'subsample': 0.18630970568769156, 'dropout_rate': 0.5777838682216713, 'n_estimators': 56, 'learning_rate': 0.07928709660452238}. Best is trial 15 with value: 0.21526209245288125.
Fold 1 IBS: 0.2788012622923502
Fold 2 IBS: 0.1792818207436949
Fold 3 IBS: 0.20339051276781625
Fold 4 IBS: 0.21868518088381278
Fold 5 IBS: 0.21558058459652885
[I 2024-04

Fold 2 IBS: 0.16568635078205915
Fold 3 IBS: 0.24223275737531913
Fold 4 IBS: 0.24198856107842706
Fold 5 IBS: 0.23125525026287133
[I 2024-04-15 16:44:01,070] Trial 38 finished with value: 0.23392553084797263 and parameters: {'subsample': 0.8179022915704841, 'dropout_rate': 0.13272164755980653, 'n_estimators': 173, 'learning_rate': 0.07288612365177065}. Best is trial 15 with value: 0.21526209245288125.
Fold 1 IBS: 0.26884425511609517
Fold 2 IBS: 0.18085364921596053
Fold 3 IBS: 0.22859592533332296
Fold 4 IBS: 0.23048206101105526
Fold 5 IBS: 0.22253306981677642
[I 2024-04-15 16:44:03,768] Trial 39 finished with value: 0.22626179209864206 and parameters: {'subsample': 0.5876128987174842, 'dropout_rate': 0.6881141432789704, 'n_estimators': 202, 'learning_rate': 0.04072308299949521}. Best is trial 15 with value: 0.21526209245288125.
Fold 1 IBS: 0.25223248598618175
Fold 2 IBS: 0.1981682459164092
Fold 3 IBS: 0.22489322453006372
Fold 4 IBS: 0.22770122870313958
Fold 5 IBS: 0.22048695288589282
[I 2

Fold 2 IBS: 0.17919879330131924
Fold 3 IBS: 0.2050086120162017
Fold 4 IBS: 0.22428306549427526
Fold 5 IBS: 0.22102485528177307
[I 2024-04-15 16:44:43,938] Trial 57 finished with value: 0.22248348521362543 and parameters: {'subsample': 0.2338181818493688, 'dropout_rate': 0.739335033868409, 'n_estimators': 140, 'learning_rate': 0.08612518078950988}. Best is trial 47 with value: 0.21386486938056484.
Fold 1 IBS: 0.24532980269644922
Fold 2 IBS: 0.21348845040668132
Fold 3 IBS: 0.21698840746960918
Fold 4 IBS: 0.2290327004310601
Fold 5 IBS: 0.22136971692396645
[I 2024-04-15 16:44:44,825] Trial 58 finished with value: 0.22524181558555326 and parameters: {'subsample': 0.1753055549049239, 'dropout_rate': 0.5462080102365253, 'n_estimators': 79, 'learning_rate': 0.04319115334522856}. Best is trial 47 with value: 0.21386486938056484.
Fold 1 IBS: 0.26773329286909525
Fold 2 IBS: 0.18016692292798644
Fold 3 IBS: 0.234355334121409
Fold 4 IBS: 0.23081623904249735
Fold 5 IBS: 0.22403143533879394
[I 2024-04

Fold 3 IBS: 0.2223499924704868
Fold 4 IBS: 0.22918954012063755
Fold 5 IBS: 0.22213633441222166
[I 2024-04-15 16:45:22,479] Trial 76 finished with value: 0.22461144613305092 and parameters: {'subsample': 0.5082505552286973, 'dropout_rate': 0.6607618331895327, 'n_estimators': 86, 'learning_rate': 0.0903014808523554}. Best is trial 64 with value: 0.21377128910644144.
Fold 1 IBS: 0.25740504494540467
Fold 2 IBS: 0.19342129698931473
Fold 3 IBS: 0.21098496634161182
Fold 4 IBS: 0.22225634861586105
Fold 5 IBS: 0.22035227228676424
[I 2024-04-15 16:45:23,783] Trial 77 finished with value: 0.2208839858357913 and parameters: {'subsample': 0.27488589720840634, 'dropout_rate': 0.8406275531062745, 'n_estimators': 107, 'learning_rate': 0.07058122348214978}. Best is trial 64 with value: 0.21377128910644144.
Fold 1 IBS: 0.277002774105818
Fold 2 IBS: 0.18852706892829543
Fold 3 IBS: 0.194491072102534
Fold 4 IBS: 0.22239803029622562
Fold 5 IBS: 0.2139914009023427
[I 2024-04-15 16:45:25,852] Trial 78 finishe

Fold 3 IBS: 0.21986105610033166
Fold 4 IBS: 0.2276997235984686
Fold 5 IBS: 0.22160135268525796
[I 2024-04-15 16:45:57,225] Trial 95 finished with value: 0.22351924440040244 and parameters: {'subsample': 0.42816774104053495, 'dropout_rate': 0.6592977171583894, 'n_estimators': 66, 'learning_rate': 0.08693594801790488}. Best is trial 64 with value: 0.21377128910644144.
Fold 1 IBS: 0.2584339011242822
Fold 2 IBS: 0.18911990790146938
Fold 3 IBS: 0.2077096426176724
Fold 4 IBS: 0.21360194311540062
Fold 5 IBS: 0.2132732733635253
[I 2024-04-15 16:45:58,486] Trial 96 finished with value: 0.21642773362447 and parameters: {'subsample': 0.1011542092953201, 'dropout_rate': 0.5430817802699582, 'n_estimators': 95, 'learning_rate': 0.08940359717281336}. Best is trial 64 with value: 0.21377128910644144.
Fold 1 IBS: 0.24834494946011265
Fold 2 IBS: 0.20875197279971033
Fold 3 IBS: 0.22691667170598823
Fold 4 IBS: 0.22507367471955336
Fold 5 IBS: 0.2199783342066774
[I 2024-04-15 16:45:59,146] Trial 97 finished

In [142]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [143]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.684
train_ibs:  0.214


#### Test

In [144]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [145]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.4136725222049048,
                                              learning_rate=0.07578621907115225,
                                              n_estimators=46, random_state=123,
                                              subsample=0.14488143846919227)

C-index score: 0.539


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.6273608789636065,
                                              learning_rate=0.07987356089292716,
                                              n_estimators=130,
                                              random_state=123,
                                              subsample=0.1018358506487617)

IBS: 0.24


In [146]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [147]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.830,1.0
ExtraSurvivalTrees,0.764,2.0
GradientBoosting,0.713,3.0
CoxElastic,0.693,4.0
CoxLasso,0.691,5.0
CoxPH,0.689,6.0
ComponentwiseGradientBoosting,0.684,7.0
CoxRidge,0.648,8.0


In [148]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.202,1.0
CoxElastic,0.206,2.0
CoxPH,0.207,4.0
CoxLasso,0.207,4.0
ExtraSurvivalTrees,0.207,4.0
ComponentwiseGradientBoosting,0.214,6.0
GradientBoosting,0.233,7.0
CoxRidge,0.236,8.0


In [149]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.544,1.0
ExtraSurvivalTrees,0.543,2.0
ComponentwiseGradientBoosting,0.539,3.0
CoxPH,0.532,6.0
CoxRidge,0.532,6.0
CoxLasso,0.532,6.0
CoxElastic,0.532,6.0
GradientBoosting,0.532,6.0


In [150]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
CoxRidge,0.229,1.5
GradientBoosting,0.229,1.5
ComponentwiseGradientBoosting,0.240,3.0
Randomsurvivalforest,0.247,4.0
ExtraSurvivalTrees,0.256,5.0
CoxLasso,0.288,6.5
CoxElastic,0.288,6.5
CoxPH,0.290,8.0


In [153]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/dfs/minmax/rent/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_dfs_minmax_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [154]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-15
